<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Torch

In [0]:
#| echo: false
#| output: asis
show_doc(create_masks)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L203){target="_blank" style="float:right; font-size:smaller"}

### create_masks

```python
def create_masks(
    x, patch_size, patch_stride, context_mask_range, target_mask_range, melt_channels_to_batch:bool=False
):
```

*Create fixed-width masks from one ratio draw shared by the batch.*

In [0]:
#| echo: false
#| output: asis
show_doc(representation_channel_stats)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L178){target="_blank" style="float:right; font-size:smaller"}

### representation_channel_stats

```python
def representation_channel_stats(
    x, max_vectors:int=1024
):
```

*Return bounded per-channel mean, std, and pairwise cosine diagnostics.*

In [0]:
#| echo: false
#| output: asis
show_doc(apply_position_masks)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L170){target="_blank" style="float:right; font-size:smaller"}

### apply_position_masks

```python
def apply_position_masks(
    positions, masks
):
```

*Gather original patch positions using one index row per batch item.*

In [0]:
#| echo: false
#| output: asis
show_doc(apply_masks)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L161){target="_blank" style="float:right; font-size:smaller"}

### apply_masks

```python
def apply_masks(
    x, masks
):
```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(JEPABlock)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L119){target="_blank" style="float:right; font-size:smaller"}

### JEPABlock

```python
def JEPABlock(
    dim, num_heads, mlp_ratio:float=4.0, qkv_bias:bool=False, qk_scale:NoneType=None, drop:float=0.0,
    attn_drop:float=0.0, act_layer:type=GELU, norm_layer:type=LayerNorm, rotary_pes:bool=False
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(PositionAwareTSTBlock)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L82){target="_blank" style="float:right; font-size:smaller"}

### PositionAwareTSTBlock

```python
def PositionAwareTSTBlock(
    d_model, n_heads, d_ff:int=256, attn_dropout:int=0, dropout:float=0.0, bias:bool=True, activation:str='gelu',
    pre_norm:bool=False, rotary_pes:bool=False
):
```

*TST block with explicit original positions for native-JEPA RoPE.*

In [0]:
#| echo: false
#| output: asis
show_doc(PositionAwareMultiHeadAttention)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L24){target="_blank" style="float:right; font-size:smaller"}

### PositionAwareMultiHeadAttention

```python
def PositionAwareMultiHeadAttention(
    dim, num_heads:int=8, qkv_bias:bool=False, qk_scale:NoneType=None, attn_drop:float=0.0, proj_drop:float=0.0,
    rotary_pes:bool=False
):
```

*Native-JEPA attention that applies RoPE using original patch positions.*

In [ ]:
def test_apply_masks_preserves_effective_batch():
    x = torch.arange(6 * 5 * 2).reshape(6, 5, 2)
    masks = torch.tensor([[0, 2, 4], [1, 2, 3], [0, 1, 4], [2, 3, 4], [0, 3, 4], [1, 3, 4]])
    masked = apply_masks(x, masks)
    assert masked.shape == (6, 3, 2)
    for batch_idx in range(x.shape[0]):
        assert torch.equal(masked[batch_idx], x[batch_idx, masks[batch_idx]])

def test_mask_ratios_do_not_collapse_across_the_effective_batch():
    torch.manual_seed(12)
    x = torch.empty(128, 3, 1800)
    observed = []
    for _ in range(12):
        targets, contexts = create_masks(x, 10, 10, (0.1, 0.4), (0.1, 0.3), True)
        target_ratio = targets.shape[1] / 180
        context_ratio = contexts.shape[1] / (180 - targets.shape[1])
        assert 0.1 - 1 / 180 <= target_ratio <= 0.3
        assert 0.1 - 1 / 180 <= context_ratio <= 0.4
        assert targets.shape[0] == contexts.shape[0] == 128 * 3
        for target_row, context_row in zip(targets, contexts):
            assert set(target_row.tolist()).isdisjoint(context_row.tolist())
        observed.append((targets.shape[1], contexts.shape[1]))
    assert len(set(observed)) > 1

def test_position_aware_rotary_attention_is_permutation_equivariant():
    torch.manual_seed(12)
    attention = PositionAwareMultiHeadAttention(16, 4, rotary_pes=True).eval()
    x = torch.randn(2, 31, 16)
    positions = torch.arange(31).expand(2, -1)
    permutation = torch.randperm(31)
    inverse = torch.argsort(permutation)
    expected = attention(x, positions=positions)
    actual = attention(
        x[:, permutation], positions=positions[:, permutation]
    )[:, inverse]
    torch.testing.assert_close(actual, expected, atol=2e-6, rtol=2e-6)

test_apply_masks_preserves_effective_batch()
test_mask_ratios_do_not_collapse_across_the_effective_batch()
test_position_aware_rotary_attention_is_permutation_equivariant()

In [0]:
#| echo: false
#| output: asis
show_doc(Encoder)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L240){target="_blank" style="float:right; font-size:smaller"}

### Encoder

```python
def Encoder(
    c_in, num_patches, patch_size, patch_stride, d_model, nhead, num_layers, use_tst_block:bool=False,
    shared_embedding:bool=True, pe_type:str='tAPE', mlp_ratio:float=4.0, qkv_bias:bool=True, qk_scale:NoneType=None,
    drop_rate:float=0.0, attn_drop_rate:float=0.0, norm_layer:type=LayerNorm, jepa:bool=True,
    embed_activation:GELU=GELU(approximate='none'), init_std:float=0.02, tokenizer_type:str='simple',
    tokenizer_kwargs:dict={}
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(mse_variance_loss)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L661){target="_blank" style="float:right; font-size:smaller"}

### mse_variance_loss

```python
def mse_variance_loss(
    pred, target_ema, representations, alpha:float=0.2
):
```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(loss_pred)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L654){target="_blank" style="float:right; font-size:smaller"}

### loss_pred

```python
def loss_pred(
    pred, target_ema, representations:NoneType=None, alpha:float=0.2
):
```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(variance_loss)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L651){target="_blank" style="float:right; font-size:smaller"}

### variance_loss

```python
def variance_loss(
    x
):
```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(Predictor)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L422){target="_blank" style="float:right; font-size:smaller"}

### Predictor

```python
def Predictor(
    num_patches, encoder_embed_dim:int=128, predictor_embed_dim:int=128, nhead:int=2, num_layers:int=1,
    use_tst_block:bool=False, pe_type:str='tAPE', mlp_ratio:float=4.0, qkv_bias:bool=True, qk_scale:NoneType=None,
    drop_rate:float=0.0, attn_drop_rate:float=0.0, norm_layer:type=LayerNorm,
    embed_activation:GELU=GELU(approximate='none'), init_std:float=0.02,
    c_in_mask_tokens:int=1, # number of channels in the encoder (if treating channels sep)
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(JEPASimpleLightning)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L670){target="_blank" style="float:right; font-size:smaller"}

### JEPASimpleLightning

```python
def JEPASimpleLightning(
    learning_rate, train_size, batch_size, n_gpus, patchtsjepa_encoder_kwargs, patchtsjepa_predictor_kwargs,
    weight_decay:float=0.04, use_weight_decay_scheduler:bool=False, final_weight_decay:float=0.4, epochs:int=100,
    optimizer_type:str='adamw', scheduler_type:str='OneCycle',
    target_mask_range:tuple=(0.05, 0.3), # the target can be up to 50% of the original x
    context_mask_range:tuple=(0.5, 1.0), # the context can be up to 80% of masked out target (1-target_mask_ratio)
    mask_block_range:tuple=(1, 30), ema_decay:float=0.996, scheduler_kwargs:dict={}, transforms:NoneType=None,
    loss_fn:function=loss_pred
):
```

*Hooks to be used in LightningModule.*

## ECG-JEPA

> Adapted from https://github.com/sehunfromdaegu/ECG_JEPA/tree/master

In [0]:
#| echo: false
#| output: asis
show_doc(ECGJEPALightning)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L1283){target="_blank" style="float:right; font-size:smaller"}

### ECGJEPALightning

```python
def ECGJEPALightning(
    encoder_kwargs, predictor_kwargs, learning_rate, train_size, batch_size, n_gpus, weight_decay:float=0.04,
    use_weight_decay_scheduler:bool=False, final_weight_decay:float=0.4, epochs:int=100, optimizer_type:str='adamw',
    scheduler_type:str='OneCycle', ema_decay:float=0.996, scheduler_kwargs:dict={}, transforms:NoneType=None
):
```

*Hooks to be used in LightningModule.*

In [0]:
#| echo: false
#| output: asis
show_doc(MaskTransformerPredictor)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L1181){target="_blank" style="float:right; font-size:smaller"}

### MaskTransformerPredictor

```python
def MaskTransformerPredictor(
    d_model:int=384, predictor_embed_dim:int=192, num_layers:int=4, nhead:int=6, mlp_ratio:float=4.0,
    qkv_bias:bool=False, qk_scale:NoneType=None, drop_rate:float=0.0, attn_drop_rate:float=0.0,
    drop_path_rate:float=0.0, norm_layer:type=LayerNorm, init_std:float=0.02, pe_type:str='sincos', c_in:int=9,
    num_patches:int=50, patch_size:int=50
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(MaskTransformer)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L1023){target="_blank" style="float:right; font-size:smaller"}

### MaskTransformer

```python
def MaskTransformer(
    d_model:int=384, num_layers:int=12, nhead:int=6, mlp_ratio:float=4.0, qkv_bias:bool=False,
    qk_scale:NoneType=None, drop_rate:float=0.0, attn_drop_rate:float=0.0, drop_path_rate:float=0.0,
    norm_layer:type=LayerNorm, init_std:float=0.02, mask_scale:tuple=(0.3, 0.5), mask_type:str='block',
    pe_type:str='sincos', c_in:int=3, num_patches:int=50, patch_size:int=50
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(Predictor_Block)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L1005){target="_blank" style="float:right; font-size:smaller"}

### Predictor_Block

```python
def Predictor_Block(
    predictor_embed_dim:int=192, depth:int=4, num_heads:int=6, mlp_ratio:float=4.0, qkv_bias:bool=False,
    qk_scale:NoneType=None, drop_rate:float=0.0, attn_drop_rate:float=0.0, drop_path_rate:float=0.0
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(Encoder_Block)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L988){target="_blank" style="float:right; font-size:smaller"}

### Encoder_Block

```python
def Encoder_Block(
    embed_dim:int=384, depth:int=12, num_heads:int=6, mlp_ratio:float=4.0, qkv_bias:bool=False,
    qk_scale:NoneType=None, drop_rate:float=0.0, attn_drop_rate:float=0.0, drop_path_rate:float=0.0,
    norm_layer:type=LayerNorm
):
```

*Base class for all neural network modules.*

Your models should also subclass this class.

Modules can also contain other Modules, allowing them to be nested in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will also have their
parameters converted when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [0]:
#| echo: false
#| output: asis
show_doc(get_2d_sincos_pos_embed)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L969){target="_blank" style="float:right; font-size:smaller"}

### get_2d_sincos_pos_embed

```python
def get_2d_sincos_pos_embed(
    embed_dim, grid_size_h, grid_size_w, cls_token:bool=False
):
```

*grid_size_h: int of the grid height*
grid_size_w: int of the grid width
return:
pos_embed: [grid_size_h*grid_size_w, embed_dim] or [1+grid_size_h*grid_size_w, embed_dim] (w/ or w/o cls_token)

In [0]:
#| echo: false
#| output: asis
show_doc(get_2d_sincos_pos_embed_from_grid)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L959){target="_blank" style="float:right; font-size:smaller"}

### get_2d_sincos_pos_embed_from_grid

```python
def get_2d_sincos_pos_embed_from_grid(
    embed_dim, grid
):
```

*Call self as a function.*

In [0]:
#| echo: false
#| output: asis
show_doc(get_1d_sincos_pos_embed_from_grid)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/jepa.py#L939){target="_blank" style="float:right; font-size:smaller"}

### get_1d_sincos_pos_embed_from_grid

```python
def get_1d_sincos_pos_embed_from_grid(
    embed_dim, pos
):
```

*embed_dim: output dimension for each position*
pos: a list of positions to be encoded: size (M,)
out: (M, D)